In [62]:
import pandas as pd

In [63]:
#Import previously used table and add columns from previous question to use for questions below

orders = pd.read_csv('ecommerce_orders.csv')
orders['Order_Date'] = pd.to_datetime(orders['Order_Date'])
orders['Ship_Date'] = pd.to_datetime(orders['Ship_Date'])
orders['Total_Before_Discount'] = (orders['Quantity'] * orders['Unit_Price'])
orders['Total_After_Discount'] = (orders['Total_Before_Discount'] * (1 - orders['Discount_Pct'] / 100)).round(2)
orders['Shipping_Days'] = (orders['Ship_Date'] - orders['Order_Date']).dt.days
orders

,Order_ID,Customer_ID,Order_Date,Ship_Date,Product,Category,Quantity,Unit_Price,Discount_Pct,Region,Payment_Method,Total_Before_Discount,Total_After_Discount,Shipping_Days
0,O001,C201,2024-01-02,2024-01-04,Laptop,Electronics,1,999.99,0,East,Credit Card,999.99,999.99,2
1,O002,C202,2024-01-03,2024-01-07,Headphones,Electronics,2,79.99,10,West,PayPal,159.98,143.98,4
2,O003,C203,2024-01-05,2024-01-06,Desk Chair,Furniture,1,349.99,0,East,Credit Card,349.99,349.99,1
3,O004,C201,2024-01-08,2024-01-09,Mouse,Electronics,3,29.99,5,East,Credit Card,89.97,85.47,1
4,O005,C204,2024-01-10,2024-01-15,Bookshelf,Furniture,1,199.99,15,South,Debit Card,199.99,169.99,5
5,O006,C205,2024-01-12,2024-01-13,Keyboard,Electronics,2,59.99,0,West,PayPal,119.98,119.98,1
6,O007,C202,2024-01-15,2024-01-22,Sofa,Furniture,1,899.99,20,West,Credit Card,899.99,719.99,7
7,O008,C206,2024-01-17,2024-01-18,Tablet,Electronics,1,499.99,10,North,Debit Card,499.99,449.99,1
8,O009,C203,2024-01-20,2024-01-21,Lamp,Furniture,4,44.99,0,East,Credit Card,179.96,179.96,1
9,O010,C207,2024-01-22,2024-01-28,Monitor,Electronics,1,349.99,5,North,PayPal,349.99,332.49,6


In [64]:
# Identify repeat customers (customers with more than one order). 

repeat_customers = orders.groupby('Customer_ID').size().reset_index(name = 'Number_of_Orders')
repeat_customers

most_purchased_product = orders.groupby(['Customer_ID', 'Product'])['Quantity'].size().reset_index(name = 'Item_Amount')
most_purchased_product

# For repeat customers only, find each one's most purchased Product using .drop_duplicates() after sorting. 
rcmpp = (most_purchased_product[most_purchased_product['Customer_ID'].isin(repeat_customers['Customer_ID'])].groupby(['Customer_ID', 'Product']).size().reset_index(name = 'Amount')).drop_duplicates(subset = 'Customer_ID')
rcmpp

# Calculate their total Revenue (Total_After_Discount), average Shipping_Days, and total number of orders. 
total_rev = orders.groupby('Customer_ID')['Total_After_Discount'].sum().reset_index(name = 'Total_Revenue')
avg_ship = orders.groupby('Customer_ID')['Shipping_Days'].mean().reset_index(name = 'Average_Shipping_Days')

combined = repeat_customers.merge(rcmpp[['Customer_ID', 'Product']], on = 'Customer_ID')
combined = combined.merge(total_rev, on = 'Customer_ID')
combined = combined.merge(avg_ship, on = 'Customer_ID')
combined = combined.sort_values('Total_Revenue', ascending = False)

# Find which repeat customer has the highest total Revenue using .idxmax(). 
cust_most_rev = combined.loc[combined['Total_Revenue'].idxmax(), 'Customer_ID']
print(f'The customer with the highest total revenue is: {cust_most_rev}')

# Calculate the cumulative Revenue per repeat customer ordered by Order_Date using .cumsum(). 
repeat_orders = orders[orders['Customer_ID'].isin(repeat_customers['Customer_ID'])].copy()
repeat_orders = repeat_orders.sort_values('Order_Date')
repeat_orders['Cumulative_Revenue'] = repeat_orders.groupby('Customer_ID')['Total_After_Discount'].cumsum()

# Then use .shift() to calculate the order-over-order Revenue change per customer. 
previous = repeat_orders.groupby('Customer_ID')['Total_After_Discount'].shift(1)
repeat_orders['Order_Revenue_Change'] = ((repeat_orders['Total_After_Discount'] - previous) / previous * 100).round(1)
repeat_orders


# Use .agg() with named aggregations to summarize per Region: total Revenue, average Shipping_Days, and number of unique customers (nunique). 
region_stats = orders.groupby('Region').agg(Total_Revenue = ('Total_After_Discount', 'sum'), Average_Shipping_Days = ('Shipping_Days', 'mean'), Number_of_Unique_Customers = ('Customer_ID', 'nunique'))

# Rank regions by total Revenue descending and use .map() with a dictionary to label regions with total Revenue > 2000 as 'High Performing' and others as 'Emerging'. 
region_stats['Rank'] = region_stats['Total_Revenue'].rank(ascending = False).astype(int)
performance_dict = {region: 'High Performing' if revenue > 2000 else 'Emerging' for region, revenue in region_stats['Total_Revenue'].items()}
region_stats['Region_Performance'] = region_stats.index.map(performance_dict)
region_stats

# Finally, label each repeat customer using a nested lambda: 'VIP' if total orders > 3, 'Regular' if 2-3 orders.
combined['Customer_Label'] = combined['Number_of_Orders'].apply(lambda x: 'VIP' if x > 3 else ('Regular' if x >= 2 else ('Only 1 Order :(' if x == 1 else 'No Orders')))
combined

The customer with the highest total revenue is: C201


,Customer_ID,Number_of_Orders,Product,Total_Revenue,Average_Shipping_Days,Customer_Label
0,C201,4,Headphones,1293.43,1.25,VIP
3,C204,2,Bookshelf,1119.98,3.50,Regular
1,C202,3,Headphones,998.92,4.00,Regular
2,C203,3,Desk Chair,700.92,1.00,Regular
8,C209,1,Desk Chair,699.98,1.00,Only 1 Order :(
9,C210,1,Sofa,674.99,6.00,Only 1 Order :(
7,C208,1,Dining Table,539.99,4.00,Only 1 Order :(
5,C206,2,Lamp,539.97,1.00,Regular
6,C207,1,Monitor,332.49,6.00,Only 1 Order :(
4,C205,2,Bookshelf,319.97,3.50,Regular
